In [ ]:
# Example 1: primary public API surface.
from opentelemetry.instrumentation.asgi import (
    OpenTelemetryMiddleware,
    collect_request_attributes,
    collect_custom_headers_attributes,
)

print("OpenTelemetryMiddleware:", OpenTelemetryMiddleware)
print("collect_request_attributes:", collect_request_attributes)
print("collect_custom_headers_attributes:", collect_custom_headers_attributes)

assert callable(OpenTelemetryMiddleware)
assert callable(collect_request_attributes)
assert callable(collect_custom_headers_attributes)


In [ ]:
# Example 2: collect_request_attributes from an ASGI scope dict.
from opentelemetry.instrumentation.asgi import collect_request_attributes

scope = {
    "type": "http",
    "http_version": "1.1",
    "method": "GET",
    "scheme": "https",
    "path": "/api/items",
    "raw_path": b"/api/items",
    "query_string": b"limit=10",
    "server": ("example.com", 443),
    "client": ("127.0.0.1", 50001),
    "headers": [(b"host", b"example.com"), (b"user-agent", b"smoke-test/1.0")],
}

attrs = collect_request_attributes(scope)
print("attrs keys:", sorted(attrs.keys()))
print("attrs sample:", {k: attrs[k] for k in list(attrs)[:5]})
assert isinstance(attrs, dict)
assert len(attrs) > 0


In [ ]:
# Example 3: instantiate OpenTelemetryMiddleware around a trivial ASGI app.
from opentelemetry.instrumentation.asgi import OpenTelemetryMiddleware

async def app(scope, receive, send):
    await send({"type": "http.response.start", "status": 200, "headers": []})
    await send({"type": "http.response.body", "body": b"ok"})

wrapped = OpenTelemetryMiddleware(app)
print("wrapped middleware:", wrapped)
print("wrapped.app is app:", wrapped.app is app)
assert wrapped.app is app
assert callable(wrapped)


In [ ]:
# Example 4: drive the middleware through one HTTP request lifecycle.
from opentelemetry.instrumentation.asgi import OpenTelemetryMiddleware

async def app(scope, receive, send):
    await send({"type": "http.response.start", "status": 204, "headers": []})
    await send({"type": "http.response.body", "body": b""})

scope = {
    "type": "http", "http_version": "1.1", "method": "GET",
    "scheme": "http", "path": "/ping", "raw_path": b"/ping",
    "query_string": b"", "headers": [(b"host", b"localhost")],
    "server": ("localhost", 80), "client": ("127.0.0.1", 1234),
}
sent = []
async def receive():
    return {"type": "http.request", "body": b"", "more_body": False}
async def send(msg):
    sent.append(msg)

wrapped = OpenTelemetryMiddleware(app)
await wrapped(scope, receive, send)
print("messages sent:", [m["type"] for m in sent])
assert any(m["type"] == "http.response.start" for m in sent)


In [ ]:
# Example 5: ASGIGetter extracts header values from ASGI carriers.
from opentelemetry.instrumentation.asgi import asgi_getter

carrier = {
    "type": "http",
    "headers": [
        (b"traceparent", b"00-0af7651916cd43dd8448eb211c80319c-b7ad6b7169203331-01"),
        (b"x-custom", b"hello"),
    ],
}

tp = asgi_getter.get(carrier, "traceparent")
custom = asgi_getter.get(carrier, "x-custom")
keys = asgi_getter.keys(carrier)
print("traceparent:", tp)
print("x-custom:", custom)
print("keys:", keys)
assert tp and tp[0].startswith("00-")
assert custom == ["hello"]
assert "traceparent" in keys
